# Lee2019 Fine-Tuning with SignalJEPA_PreLocal

Downstream fine-tuning on **Lee2019 Dataset**.

- Dataset: Lee2019 (MOABB)
- EEG channels only; bandpass 0.5–40 Hz; resample to 128 Hz
- 5-fold stratified within-subject cross-validation
- Model: SignalJEPA_PreLocal
- Pretrained weights: Hugging Face 16s-60 checkpoint
- Fine-tuning scheme: **new-pre-local** (frozen pretrained backbone, train only new downstream layers)
- Training: Braindecode EEGClassifier (no manual loop)

Note: dataset split, preprocessing, downstream window, model family, and fine-tuning strategy follow the S-JEPA paper design. Optimizer hyperparameters in this notebook are implementation defaults.

# 1. Setup

## 1.1. Import Libraries

In [1]:
import os
import random
import sys
from pathlib import Path
import platform
import json
import hashlib
from datetime import datetime

import pandas as pd
import numpy as np
from numpy import multiply

import torch
from torch.utils.data import Subset

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import (
    Preprocessor,
    preprocess,
    create_windows_from_events,
    exponential_moving_standardize,
)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from skorch.callbacks import (
  EarlyStopping,
  Callback,
)
from skorch.helper import predefined_split

import builtins

from moabb.datasets import Lee2019_SSVEP, Lee2019_ERP, Lee2019_MI

import mne
mne.set_log_level("WARNING")

print("All imports loaded successfully.")

All imports loaded successfully.


/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2. Runtime & Path Validation

In [2]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().parent
print(f"\nWorking directory: {WORKING_DIR}")

Runtime Environment:
  - Python: 3.11.15 | packaged by conda-forge | (main, Mar  5 2026, 16:45:40) [GCC 14.3.0]
  - Platform: Linux-6.14.0-37-generic-x86_64-with-glibc2.41

Working directory: /home/vegorov/Repos/eeg_jepa_research


# 2. Configuration

## 2.1. Config

In [3]:
CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "lee-2019-fine-tuning"),

    # Reproducibility
    "seed": None, # None for random seed, or specify integer
    "set_seed": False,

    # Preprocessing (paper-aligned)
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,

    # Preprocessing (implementation defaults, not paper claims)
    "set_eeg_reference": True,
    "convert_to_uV": True,
    "normalize": False,
    "normalization_method": None, # exponential_moving, exponential_moving_slow, zscore, robust, demean
    "learning_rate": None, # None = default skorch learning rate 0.01

    # Subjects (None = auto-select last 7, or specify list of subject IDs)
    "subjects_to_use": None,

    # Paradigm settings
    "paradigm": "ERP", # 'SSVEP', 'MI', 'ERP'

    # Cross-validation (paper-aligned structure)
    "cv_folds": 5,
    "val_split": 0.2,

    # Training (implementation defaults, not paper claims)
    "batch_size": 32,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "optimizer": "Adam", # Adam, AdamW
    "criterion": None, # None, CrossEntropyLoss, MSELoss, etc. 

    # Pretrained weights (Hugging Face 16s-60 checkpoint)
    "pretrained_url": "https://huggingface.co/braindecode/SignalJEPA/resolve/main/signal-jepa_16s-60_adeuwv4s.pth",

    # Device: "auto" | "cpu" | "cuda" | "mps"
    "device": "auto",
}

# Output verbosity flags (set to True for more detailed output)
LOG_COMPACT = False
PRINT_MODEL_SUMMARY = True
PRINT_FREEZE_DETAILS = True
PRINT_FOLD_CLASS_COUNTS = True


In [4]:
PARADIGM_CONFIGS = {
    "SSVEP": {
        "n_classes": 4,
        "base_trial_duration_s": 4.0,
        "requested_trial_duration_s": 4.19,
        "window_samples_size": 536,
    },
    "MI": {
        "n_classes": 2,
        "base_trial_duration_s": 4.0,
        "requested_trial_duration_s": 4.19,
        "window_samples_size": 536,
    },
    "ERP": {
        "n_classes": 2,
        "base_trial_duration_s": 1.0,
        "requested_trial_duration_s": 1.19,
        "window_samples_size": 152,
    },
}

## 2.2. Create Artifact Directory

In [5]:
def create_run_id():
    # Generate unique run ID from timestamp + config hash.
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

In [6]:
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["paradigm"] / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 2.3. Initialize Logger

In [7]:
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)

    message = sep.join(str(arg) for arg in args)

    # Preserve visual spacing for prints like print("\n...")
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            sys.__stdout__.write(text) # type: ignore
            if flush:
                sys.__stdout__.flush() # type: ignore
        else:
            file.write(text)
            if flush and hasattr(file, "flush"):
                file.flush()

    # Apply leading blank lines first without timestamps
    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _LOG_FILE_HANDLE.write(blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _LOG_FILE_HANDLE.write(stamped + end)
    else:
        # If only newlines were printed, preserve end behavior without adding a timestamp
        _write_target(end)
        _LOG_FILE_HANDLE.write(end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

## 2.4. Device Configuration

In [8]:
def resolve_device(device):
    """Resolve the compute device from config."""
    if device == "cpu":
        return torch.device("cpu")
    if device == "cuda":
        return torch.device("cuda")
    if device == "mps":
        return torch.device("mps")

    # "auto": prefer MPS > CUDA > CPU
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = resolve_device(CONFIG["device"])
print(f"Using device: {DEVICE}")

[2026-04-10 11:45:06] Using device: cpu


## 2.5. Deterministic Seeding

In [9]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    _ = worker_id
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None

if CONFIG["set_seed"]:
    if BASE_SEED is None:
        raise ValueError("Seed control is enabled but CONFIG['seed'] is None. Please specify a seed.")

    seed_everything(BASE_SEED)
    print(f"Seed control enabled: {CONFIG['set_seed']}")
    print(f"Base seed: {BASE_SEED}")
    print(f"Seed initialized: {BASE_SEED}")
else:
    print("Seed control disabled (CONFIG['set_seed'] = False)")

[2026-04-10 11:45:06] Seed control disabled (CONFIG['set_seed'] = False)


## 2.6. Save Configuration

In [10]:
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
for key in sorted(CONFIG.keys()):
    print(f"  {key}: {CONFIG[key]}")
print("=" * 70)

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Config saved to: {config_path}")

[2026-04-10 11:45:06] ======================================================================
[2026-04-10 11:45:06] CONFIGURATION
[2026-04-10 11:45:06] ======================================================================
[2026-04-10 11:45:06]   artifact_dir: /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning
[2026-04-10 11:45:06]   bandpass_high: 40.0
[2026-04-10 11:45:06]   bandpass_low: 0.5
[2026-04-10 11:45:06]   batch_size: 32
[2026-04-10 11:45:06]   convert_to_uV: True
[2026-04-10 11:45:06]   criterion: None
[2026-04-10 11:45:06]   cv_folds: 5
[2026-04-10 11:45:06]   device: auto
[2026-04-10 11:45:06]   early_stopping_patience: 50
[2026-04-10 11:45:06]   learning_rate: None
[2026-04-10 11:45:06]   n_epochs: 5000
[2026-04-10 11:45:06]   normalization_method: None
[2026-04-10 11:45:06]   normalize: False
[2026-04-10 11:45:06]   optimizer: Adam
[2026-04-10 11:45:06]   paradigm: ERP
[2026-04-10 11:45:06]   pretrained_url: https://huggingface.co/braindecode/SignalJEP

# 3. Prepare Data

## 3.1. Derived Constants

In [11]:
def current_paradigm_config(paradigm, paradigm_configs):
    if paradigm not in ["SSVEP", "ERP", "MI"]:
        raise ValueError(f"Unsupported paradigm: {paradigm}. Must be one of 'SSVEP', 'ERP', 'MI'.")

    return (
        paradigm_configs[paradigm]["n_classes"],
        paradigm_configs[paradigm]["base_trial_duration_s"],
        paradigm_configs[paradigm]["requested_trial_duration_s"],
        paradigm_configs[paradigm]["window_samples_size"],
    )

In [12]:
def compute_epoch_window(sfreq, base_trial_duration_s, target_trial_duration_s):
    """Compute the exact downstream window and stop offset from CONFIG."""
    sfreq = float(sfreq)
    target_trial_duration_s = float(target_trial_duration_s)

    window_size_samples = round(target_trial_duration_s * sfreq)

    base_trial_duration_s = float(base_trial_duration_s)
    base_trial_samples = round(base_trial_duration_s * sfreq)
    trial_stop_offset_samples = window_size_samples - base_trial_samples
    effective_duration = window_size_samples / sfreq

    print("Epoch window configuration:")
    print(f"  Target sfreq:                {sfreq:.0f} Hz")
    print(f"  Target duration:             {target_trial_duration_s:.6f} s")
    print(f"  Window samples (rounded):    {window_size_samples}")
    print(f"  Base trial samples ({base_trial_duration_s:.1f} s):  {base_trial_samples}")
    print(f"  Trial stop offset samples:   {trial_stop_offset_samples}")
    print(f"  Effective window duration:   {effective_duration:.6f} s")

    return window_size_samples, base_trial_samples, trial_stop_offset_samples

In [13]:
TARGET_N_CLASSES, BASE_TRIAL_DURATION_S, TARGET_TRIAL_DURATION_S, EXPECTED_WINDOW_SAMPLE_SIZE = current_paradigm_config(CONFIG["paradigm"], PARADIGM_CONFIGS)

WINDOW_SAMPLES, BASE_TRIAL_SAMPLES, TRIAL_STOP_OFFSET_SAMPLES = compute_epoch_window(
    sfreq=CONFIG["sfreq"],
    base_trial_duration_s=BASE_TRIAL_DURATION_S,
    target_trial_duration_s=TARGET_TRIAL_DURATION_S,
)

[2026-04-10 11:45:06] Epoch window configuration:
[2026-04-10 11:45:06]   Target sfreq:                128 Hz
[2026-04-10 11:45:06]   Target duration:             1.190000 s
[2026-04-10 11:45:06]   Window samples (rounded):    152
[2026-04-10 11:45:06]   Base trial samples (1.0 s):  128
[2026-04-10 11:45:06]   Trial stop offset samples:   24
[2026-04-10 11:45:06]   Effective window duration:   1.187500 s


## 3.2. Subject Selection

In [14]:
def select_last_subjects(dataset, n_last=7):
    """Select the last n subjects required for downstream evaluation."""

    print("Dataset code:", dataset.code)
    print("Subjects Total:", len(dataset.subject_list))
    print("Events:", dataset.event_id)
    print("Interval:", dataset.interval)
    print("Paradigm:", dataset.paradigm)

    subjects = sorted(int(s) for s in dataset.subject_list)

    if len(subjects) < n_last:
        raise RuntimeError(f"Dataset has {len(subjects)} subjects, need at least {n_last}.")
    return subjects[-n_last:]

In [15]:
if CONFIG["paradigm"] == "SSVEP":
    LEE_DATASET = Lee2019_SSVEP() # type: ignore
elif CONFIG["paradigm"] == "MI":
    LEE_DATASET = Lee2019_MI() # type: ignore
elif CONFIG["paradigm"] == "ERP":
    LEE_DATASET = Lee2019_ERP() # type: ignore

if CONFIG["subjects_to_use"] is not None:
    SUBJECTS = CONFIG["subjects_to_use"]
    print(f"Using specified subjects from config: {SUBJECTS}")
else:
    SUBJECTS = select_last_subjects(LEE_DATASET, n_last=7)

print(f"Selected subjects: {SUBJECTS}")
print(f"Expected evaluation folds: {len(SUBJECTS) * CONFIG['cv_folds']}")

DATASET_NAME = f"Lee2019_{CONFIG['paradigm']}"
print(f"\nBuilding MOABBDataset for {DATASET_NAME}...")
DATASET = MOABBDataset(
    dataset_name=DATASET_NAME,
    subject_ids=SUBJECTS,
    dataset_kwargs={"train_run": True, "test_run": False},
)
print(f"MOABBDataset recordings loaded: {len(DATASET.datasets)}")

# event mapping diagnostics
event_id = dict(sorted(LEE_DATASET.event_id.items(), key=lambda kv: kv[1]))
label_map = {code: idx for idx, (_, code) in enumerate(event_id.items())}
first_ann = sorted(set(DATASET.datasets[0].raw.annotations.description))
print(f"Discovered annotation descriptions (first recording): {first_ann}")
print(f"Dataset event_id mapping: {event_id}")
print(f"Label map (event_code -> class_index): {label_map}")

[2026-04-10 11:45:06] Dataset code: Lee2019-ERP
[2026-04-10 11:45:06] Subjects Total: 54
[2026-04-10 11:45:06] Events: {'Target': 1, 'NonTarget': 2}
[2026-04-10 11:45:06] Interval: [0.0, 1.0]
[2026-04-10 11:45:06] Paradigm: p300
[2026-04-10 11:45:06] Selected subjects: [48, 49, 50, 51, 52, 53, 54]
[2026-04-10 11:45:06] Expected evaluation folds: 35

[2026-04-10 11:45:06] Building MOABBDataset for Lee2019_ERP...
[2026-04-10 11:46:51] MOABBDataset recordings loaded: 14
[2026-04-10 11:46:51] Discovered annotation descriptions (first recording): [np.str_('NonTarget'), np.str_('Target')]
[2026-04-10 11:46:51] Dataset event_id mapping: {'Target': 1, 'NonTarget': 2}
[2026-04-10 11:46:51] Label map (event_code -> class_index): {1: 0, 2: 1}


## 3.3. Filter and Resample Data

In [16]:
def channel_zscore(data, eps=1e-8):
    mean = data.mean(axis=-1, keepdims=True)
    std = data.std(axis=-1, keepdims=True)
    return (data - mean) / (std + eps)

def channel_robust_scale(data, eps=1e-8):
    """Per-recording, per-channel robust scaling using median and MAD."""
    median = np.median(data, axis=-1, keepdims=True)
    mad = np.median(np.abs(data - median), axis=-1, keepdims=True)
    return (data - median) / (mad + eps)

def channel_demean(data):
    """Per-recording, per-channel mean removal only."""
    mean = data.mean(axis=-1, keepdims=True)
    return data - mean

def get_preprocessors(sfreq, bandpass_low, bandpass_high, set_eeg_reference, convert_to_uV, normalize, normalization_method):
    """Build Braindecode preprocessors in memory-efficient order.

    Preprocessing steps:
      1. Pick EEG channels.
      2. Set EEG reference to average.
      3. Convert V -> uV (scale raw voltage to microvolts). [OPTIONAL]
      4. Bandpass filter.
      5. Resample to the target sampling frequency.
      6. Normalisation. [OPTIONAL]
    """

    preprocessors = [Preprocessor("pick", picks="eeg")]

    if set_eeg_reference:
        preprocessors.append(Preprocessor("set_eeg_reference", ref_channels="average"))

    if convert_to_uV:
        preprocessors.append(Preprocessor(lambda data: multiply(data, 1e6)))

    preprocessors.append(Preprocessor("filter", l_freq=bandpass_low, h_freq=bandpass_high, method="iir"))
    preprocessors.append(Preprocessor("resample", sfreq=sfreq))

    if normalize:
        if normalization_method == "exponential_moving":
            preprocessors.append(
                Preprocessor(
                    exponential_moving_standardize,
                    factor_new=1e-3,
                    init_block_size=1000,
                )
            )
        elif normalization_method == "exponential_moving_slow":
            preprocessors.append(
                Preprocessor(
                    exponential_moving_standardize,
                    factor_new=1e-4,
                    init_block_size=1000,
                )
            )
        elif normalization_method == "zscore":
            preprocessors.append(Preprocessor(channel_zscore))
        elif normalization_method == "robust":
            preprocessors.append(Preprocessor(channel_robust_scale))
        elif normalization_method == "demean":
            preprocessors.append(Preprocessor(channel_demean))
        else:
            pass

    return preprocessors

## 3.4. Channel Info and Validation

In [17]:
def get_channel_info_from_one_recording(dataset):
    """Extract channel metadata from the first preprocessed recording."""
    first_raw = dataset.datasets[0].raw
    return first_raw.info["chs"], len(first_raw.ch_names), list(first_raw.ch_names)

In [18]:
def validate_channel_consistency(dataset):
    """Validate all recordings share channel count/order after preprocessing."""
    checked_runs = 0
    unique_channel_counts = set()
    reference_names = None
    inconsistent_runs = []

    for ds in dataset.datasets:
        checked_runs += 1
        raw = ds.raw
        ch_names = tuple(raw.ch_names)
        unique_channel_counts.add(len(ch_names))

        if reference_names is None:
            reference_names = ch_names
            continue

        if ch_names != reference_names:
            inconsistent_runs.append(
                {
                    "run": str(ds.description.to_dict()),
                    "n_channels": len(ch_names),
                    "first_channels": list(ch_names[:10]),
                }
            )

    channel_order_consistent = len(inconsistent_runs) == 0
    print("Channel consistency summary:")
    print(f"  Checked runs: {checked_runs}")
    print(f"  Unique channel counts: {sorted(unique_channel_counts)}")
    print(f"  Channel order consistent: {channel_order_consistent}")

    if inconsistent_runs:
        print(f"  Inconsistent examples: {inconsistent_runs[:2]}")

    return {
        "checked_runs": checked_runs,
        "unique_channel_counts": sorted(unique_channel_counts),
        "channel_order_consistent": channel_order_consistent,
    }

In [19]:
print("Applying Braindecode preprocessors to MOABBDataset...")
preprocessors = get_preprocessors(
    sfreq=CONFIG["sfreq"],
    bandpass_low=CONFIG["bandpass_low"],
    bandpass_high=CONFIG["bandpass_high"],
    set_eeg_reference=CONFIG["set_eeg_reference"],
    convert_to_uV=CONFIG["convert_to_uV"],
    normalize=CONFIG["normalize"],
    normalization_method=CONFIG["normalization_method"],
)

# n_jobs=1 is intentional: each parallel worker would load a full copy of the
# dataset into memory, causing large RAM spikes on local machines.
# Sequential processing (n_jobs=1) is slower but keeps peak memory manageable.
preprocess(DATASET, preprocessors, n_jobs=1)
print("Preprocessing complete.")

# Post-preprocessing validation
print("\nPost-preprocessing validation")
print(f"  Recordings in DATASET: {len(DATASET.datasets)}")
subject_ids = sorted(set(str(ds.description["subject"]) for ds in DATASET.datasets))
print(f"  Unique subject IDs:    {subject_ids}")
first_raw = DATASET.datasets[0].raw
print(f"  sfreq (first rec):     {first_raw.info['sfreq']} Hz")
print(f"  EEG channel count:     {len(first_raw.ch_names)}")
print(f"  First 10 channels:     {list(first_raw.ch_names[:10])}")

[2026-04-10 11:46:51] Applying Braindecode preprocessors to MOABBDataset...


/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:76: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(
/home/vegorov/Repos/eeg_jepa_research/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:74: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


[2026-04-10 11:47:19] Preprocessing complete.

[2026-04-10 11:47:19] Post-preprocessing validation
[2026-04-10 11:47:19]   Recordings in DATASET: 14
[2026-04-10 11:47:19]   Unique subject IDs:    ['48', '49', '50', '51', '52', '53', '54']
[2026-04-10 11:47:19]   sfreq (first rec):     128.0 Hz
[2026-04-10 11:47:19]   EEG channel count:     62
[2026-04-10 11:47:19]   First 10 channels:     ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2']


In [20]:
# validate_channel_consistency(DATASET)
CHS_INFO, N_CHANNELS, CH_NAMES = get_channel_info_from_one_recording(DATASET)

print(f"EEG channel #: {N_CHANNELS}")
print(f"First 10 EEG channels: {CH_NAMES[:10]}")

[2026-04-10 11:47:19] EEG channel #: 62
[2026-04-10 11:47:19] First 10 EEG channels: ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2']


## 3.5. Subject Preprocessing

In [21]:
def build_windows_dataset(dataset, window_size_samples, trial_stop_offset_samples):
    """Create event-based windows from preprocessed MOABBDataset."""
    trial_start_offset_samples = 0

    print(f"Requested window samples: {window_size_samples}")
    print(f"Base trial samples:       {BASE_TRIAL_SAMPLES}")
    print(f"Stop offset samples:      {trial_stop_offset_samples}")

    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=trial_start_offset_samples,
        trial_stop_offset_samples=trial_stop_offset_samples,
        window_size_samples=window_size_samples,
        window_stride_samples=window_size_samples,
        drop_last_window=False,
        preload=True,
    )

    return windows_dataset

In [22]:
def get_targets_from_dataset(dataset):
    """Extract integer labels from a Braindecode dataset/subset."""
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

In [23]:
def get_subject_windows(windows_dataset, subject_ids):
    """Split windows by subject and ensure all requested subjects are available."""
    split_by_subject = windows_dataset.split("subject")
    subject_windows = {}

    for sid in subject_ids:
        key = str(sid)
        if key in split_by_subject:
            subject_windows[sid] = split_by_subject[key]
        elif sid in split_by_subject:
            subject_windows[sid] = split_by_subject[sid]
        else:
            raise RuntimeError(f"Subject {sid} not found in windows split keys: {list(split_by_subject.keys())[:10]}")

    return subject_windows

In [24]:
# raw = DATASET.datasets[0].raw
# data = raw.get_data()
# print(f"raw mean_abs={np.mean(np.abs(data)):.6e}")
# print(f"raw std={np.std(data):.6e}")
# print("raw first5 channel std=", np.std(data, axis=1)[:5])

WINDOWS_DATASET = build_windows_dataset(
    DATASET,
    window_size_samples=WINDOW_SAMPLES,
    trial_stop_offset_samples=TRIAL_STOP_OFFSET_SAMPLES,
)

# Global label diagnostics
ALL_LABELS = get_targets_from_dataset(WINDOWS_DATASET)
all_counts = np.bincount(ALL_LABELS, minlength=TARGET_N_CLASSES)
print(f"Global class counts: {all_counts.tolist()}")
print(f"Chance level ({TARGET_N_CLASSES}-class): {1.0 / TARGET_N_CLASSES:.2f}")

# Window/sample diagnostics
sample0 = WINDOWS_DATASET[0]
x0 = sample0[0]
y0 = sample0[1]
print(f"One window shape: {tuple(x0.shape)}")
print(f"One window label: {int(y0)}")
print(f"Window sample length expected={WINDOW_SAMPLES}, got={x0.shape[-1]}")

SUBJECT_WINDOWS = get_subject_windows(WINDOWS_DATASET, SUBJECTS)
print(f"Subjects in window split: {list(SUBJECT_WINDOWS.keys())}")

[2026-04-10 11:47:19] Requested window samples: 152
[2026-04-10 11:47:19] Base trial samples:       128
[2026-04-10 11:47:19] Stop offset samples:      24
[2026-04-10 11:47:19] Global class counts: [23100, 4620]
[2026-04-10 11:47:19] Chance level (2-class): 0.50
[2026-04-10 11:47:19] One window shape: (62, 152)
[2026-04-10 11:47:19] One window label: 0
[2026-04-10 11:47:19] Window sample length expected=152, got=152
[2026-04-10 11:47:19] Subjects in window split: [48, 49, 50, 51, 52, 53, 54]


In [25]:
def summarize_subject_windows(subject_windows, n_classes):
    """Summarize per-subject window counts and class balance."""
    rows = []
    for subject_id in sorted(subject_windows):
        ds = subject_windows[subject_id]
        y = get_targets_from_dataset(ds)
        counts = np.bincount(y, minlength=n_classes)
        sample = ds[0]
        x = sample[0]

        print(
            f"  Subject {subject_id}: n_windows={len(ds)}, "
            f"window_shape={tuple(x.shape)}, class_counts={counts.tolist()}"
        )
        rows.append(
            {
                "subject_id": int(subject_id),
                "n_windows": int(len(ds)),
                "n_channels": int(x.shape[0]),
                "n_times": int(x.shape[1]),
                "class_counts": counts.tolist(),
            }
        )

    return pd.DataFrame(rows)

In [26]:
print("Summarizing per-subject windows...")
SUBJECT_TRIALS_SUMMARY = summarize_subject_windows(SUBJECT_WINDOWS, TARGET_N_CLASSES)
print(SUBJECT_TRIALS_SUMMARY)

# Keep compatibility with downstream reporting code paths.
SUBJECT_TRIALS = {sid: (None, get_targets_from_dataset(ds)) for sid, ds in SUBJECT_WINDOWS.items()}

[2026-04-10 11:47:19] Summarizing per-subject windows...
[2026-04-10 11:47:19]   Subject 48: n_windows=3960, window_shape=(62, 152), class_counts=[3300, 660]
[2026-04-10 11:47:19]   Subject 49: n_windows=3960, window_shape=(62, 152), class_counts=[3300, 660]
[2026-04-10 11:47:19]   Subject 50: n_windows=3960, window_shape=(62, 152), class_counts=[3300, 660]
[2026-04-10 11:47:19]   Subject 51: n_windows=3960, window_shape=(62, 152), class_counts=[3300, 660]
[2026-04-10 11:47:19]   Subject 52: n_windows=3960, window_shape=(62, 152), class_counts=[3300, 660]
[2026-04-10 11:47:19]   Subject 53: n_windows=3960, window_shape=(62, 152), class_counts=[3300, 660]
[2026-04-10 11:47:19]   Subject 54: n_windows=3960, window_shape=(62, 152), class_counts=[3300, 660]
[2026-04-10 11:47:19]    subject_id  n_windows  n_channels  n_times class_counts
0          48       3960          62      152  [3300, 660]
1          49       3960          62      152  [3300, 660]
2          50       3960          62 

## 3.6. Validation and Summary

In [27]:
def inspect_preprocessing_sanity(windows_dataset):
    """Quick sanity checks on one window after preprocessing."""
    sample = windows_dataset[0]
    x = sample[0]
    y = sample[1]
    x_np = np.asarray(x)

    print("Preprocessing sanity checks:")
    print(f"  One sample label: {int(y)}")
    print(f"  One sample shape: {tuple(x_np.shape)}")
    print(f"  Mean(abs(signal)): {float(np.mean(np.abs(x_np))):.6e}")
    print(f"  Mean(signal):      {float(np.mean(x_np)):.6e}")
    print(f"  Std(signal):       {float(np.std(x_np)):.6e}")

In [28]:
print("Window validation summary:")

assert len(WINDOWS_DATASET) > 0, "No windows created."
sample0 = WINDOWS_DATASET[0]
x0 = sample0[0]
assert x0.shape[-1] == WINDOW_SAMPLES, (
    f"Window length mismatch: got {x0.shape[-1]} expected {WINDOW_SAMPLES}"
)
assert WINDOW_SAMPLES == EXPECTED_WINDOW_SAMPLE_SIZE, f"Expected {EXPECTED_WINDOW_SAMPLE_SIZE} samples, got {WINDOW_SAMPLES}"

all_labels = get_targets_from_dataset(WINDOWS_DATASET)
assert np.isin(all_labels, np.arange(TARGET_N_CLASSES)).all(), (
    f"Invalid labels found: {np.unique(all_labels)}"
)

for sid, ds in SUBJECT_WINDOWS.items():
    y_sid = get_targets_from_dataset(ds)
    counts = np.bincount(y_sid, minlength=TARGET_N_CLASSES)
    print(f"  Subject {sid}: n_windows={len(ds)} class_counts={counts.tolist()}")

effective_duration = WINDOW_SAMPLES / CONFIG["sfreq"]

print(f"\nValidated sampling rate target: {CONFIG['sfreq']} Hz")
print(f"Validated window sample count:   {WINDOW_SAMPLES}")
print(f"Validated window duration:       {effective_duration:.4f} s")
print(f"Validated channel count:         {N_CHANNELS}")
print(f"Total windows retained:          {len(WINDOWS_DATASET)}")

inspect_preprocessing_sanity(WINDOWS_DATASET)

[2026-04-10 11:47:20] Window validation summary:
[2026-04-10 11:47:20]   Subject 48: n_windows=3960 class_counts=[3300, 660]
[2026-04-10 11:47:20]   Subject 49: n_windows=3960 class_counts=[3300, 660]
[2026-04-10 11:47:20]   Subject 50: n_windows=3960 class_counts=[3300, 660]
[2026-04-10 11:47:20]   Subject 51: n_windows=3960 class_counts=[3300, 660]
[2026-04-10 11:47:20]   Subject 52: n_windows=3960 class_counts=[3300, 660]
[2026-04-10 11:47:20]   Subject 53: n_windows=3960 class_counts=[3300, 660]
[2026-04-10 11:47:20]   Subject 54: n_windows=3960 class_counts=[3300, 660]

[2026-04-10 11:47:20] Validated sampling rate target: 128 Hz
[2026-04-10 11:47:20] Validated window sample count:   152
[2026-04-10 11:47:20] Validated window duration:       1.1875 s
[2026-04-10 11:47:20] Validated channel count:         62
[2026-04-10 11:47:20] Total windows retained:          27720
[2026-04-10 11:47:20] Preprocessing sanity checks:
[2026-04-10 11:47:20]   One sample label: 0
[2026-04-10 11:47:20

# 4. Model

## 4.1. Build Model

In [29]:
def build_model():
    """Instantiate SignalJEPA_PreLocal."""
    # actual window duration
    trial_s = (WINDOW_SAMPLES / CONFIG["sfreq"])
    if not LOG_COMPACT:
        print(f"Building model with input window duration: {trial_s:.4f} s")
    model = SignalJEPA_PreLocal(
        sfreq=CONFIG["sfreq"],
        input_window_seconds=trial_s,
        chs_info=CHS_INFO,
        n_outputs=TARGET_N_CLASSES,
    )
    return model


In [30]:
# Verify once that the model builds without error
test_model = build_model()
total_p = sum(p.numel() for p in test_model.parameters())
trainable_p = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print("SignalJEPA_PreLocal instantiated successfully.")
print(f"  Total parameters:     {total_p:,}")
print(f"  Trainable parameters: {trainable_p:,}")
if PRINT_MODEL_SUMMARY:
    print(test_model)
del test_model


[2026-04-10 11:47:20] Building model with input window duration: 1.1875 s
[2026-04-10 11:47:20] SignalJEPA_PreLocal instantiated successfully.
[2026-04-10 11:47:20]   Total parameters:     14,606
[2026-04-10 11:47:20]   Trainable parameters: 14,606
[2026-04-10 11:47:20] ======================================================================================================================================================
Layer (type (var_name):depth-idx)                  Input Shape               Output Shape              Param #                   Kernel Shape
SignalJEPA_PreLocal (SignalJEPA_PreLocal)          [1, 62, 152]              [1, 2]                    --                        --
├─Sequential (spatial_conv): 1-1                   [1, 62, 152]              [1, 4, 152]               --                        --
│    └─Rearrange (0): 2-1                          [1, 62, 152]              [1, 1, 62, 152]           --                        --
│    └─Conv2d (1): 2-2                  

## 4.2. Load Pretrained Weights from Hugging Face

In [31]:
def download_pretrained_state_dict(url):
    """Download Signal-JEPA pretrained weights from Hugging Face."""
    print("Initializing from 16s-60 pretrained checkpoint (Hugging Face):")
    print(f"  {url}")
    sd = torch.hub.load_state_dict_from_url(url, map_location="cpu")
    print(f"  Downloaded {len(sd)} keys")
    return sd

In [32]:
def load_pretrained_weights(model, state_dict):
    """
    Filter transformer and pos_encoder keys (not used by PreLocal downstream) and load encoder weights.
    
    Strictly validates that only expected downstream layers are missing from the pretrained weights.
    Raises an error if unexpected keys remain or if missing keys do not match downstream layers.
    """
    filtered = {
        k: v for k, v in state_dict.items()
        if not (k.startswith("transformer.") or k.startswith("pos_encoder."))
    }
    n_filtered_transformer = sum(1 for k in state_dict if k.startswith("transformer."))
    n_filtered_pos_encoder = sum(1 for k in state_dict if k.startswith("pos_encoder."))
    
    missing_keys, unexpected_keys = model.load_state_dict(filtered, strict=False)
    
    if not LOG_COMPACT:
        print("Pretrained weight loading:")
        print(f"  Downloaded keys:        {len(state_dict)}")
        print(f"  Filtered (transformer): {n_filtered_transformer}")
        print(f"  Filtered (pos_encoder): {n_filtered_pos_encoder}")
        print(f"  Keys to load:           {len(filtered)}")
        print(f"  Missing keys:           {len(missing_keys)}")
        print(f"  Unexpected keys:        {len(unexpected_keys)}")
        print(f"  Missing preview:        {missing_keys[:6]}")
        print(f"  Unexpected preview:     {unexpected_keys[:6]}")

    return {
        "downloaded_keys": len(state_dict),
        "filtered_transformer": n_filtered_transformer,
        "filtered_pos_encoder": n_filtered_pos_encoder,
        "loaded_keys": len(filtered),
        "missing_keys": list(missing_keys),
        "unexpected_keys": list(unexpected_keys),
    }


In [33]:
# Download once — reused for every fold
PRETRAINED_STATE_DICT = download_pretrained_state_dict(CONFIG["pretrained_url"])

[2026-04-10 11:47:20] Initializing from 16s-60 pretrained checkpoint (Hugging Face):
[2026-04-10 11:47:20]   https://huggingface.co/braindecode/SignalJEPA/resolve/main/signal-jepa_16s-60_adeuwv4s.pth
[2026-04-10 11:47:20]   Downloaded 179 keys


## 4.3. Freeze Encoder

In [34]:
class WeightsLoader(Callback):
    def __init__(self, url, strict=False):
        self.url = url
        self.strict = strict

    def on_train_begin(self, net, X=None, y=None, **kwargs):
        state_dict = torch.hub.load_state_dict_from_url(url=self.url)
        net.module_.load_state_dict(state_dict, strict=self.strict)

In [35]:
# Internal flag: print freeze summary only once per run
_freeze_logged = False

def freeze_encoder(model):
    """
    Freeze all encoder layers; keep only spatial_conv and final_layer trainable.
    
    Implements 'new-pre-local' strategy: all pretrained components frozen,
    only the newly introduced downstream layers trained.
    """
    global _freeze_logged
    for name, param in model.named_parameters():
        param.requires_grad = False
    
    trainable_names = []
    for name, param in model.named_parameters():
        if 'spatial_conv' in name or 'final_layer' in name:
            param.requires_grad = True
            trainable_names.append(name)
    
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    assert all(("spatial_conv" in n or "final_layer" in n) for n in trainable_names), (
        "Found trainable parameters outside spatial_conv/final_layer."
    )

    if not _freeze_logged:
        print(f"Trainable modules: spatial_conv + final_layer ({trainable:,} / {total:,} params, {100 * trainable / total:.1f}%)")
        if PRINT_FREEZE_DETAILS:
            for tname in sorted(trainable_names):
                print(f"  - {tname}")
        _freeze_logged = True

# 5. Training

## 5.1. EEGClassifier Fold Runner

In [36]:
def run_single_fold(fold_id, subject_id, subject_dataset, idx_train, idx_val, idx_test):
    """Train and evaluate one fold using EEGClassifier on dataset subsets."""
    if CONFIG["set_seed"]:
        fold_seed = BASE_SEED + int(subject_id) * 100 + int(fold_id) # type: ignore
        seed_everything(fold_seed)
    else:
        fold_seed = None

    train_set = Subset(subject_dataset, idx_train.tolist())
    valid_set = Subset(subject_dataset, idx_val.tolist())
    test_set = Subset(subject_dataset, idx_test.tolist())

    y_all = get_targets_from_dataset(subject_dataset)
    y_train = y_all[idx_train]
    y_val = y_all[idx_val]
    y_test = y_all[idx_test]

    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    val_counts = np.bincount(y_val, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    if PRINT_FOLD_CLASS_COUNTS:
        print(f"    Train class counts: {train_counts.tolist()}")
        print(f"    Val class counts:   {val_counts.tolist()}")
        print(f"    Test class counts:  {test_counts.tolist()}")

    model = build_model()
    load_pretrained_weights(model, PRETRAINED_STATE_DICT)
    freeze_encoder(model)

    callbacks = [
        (
            "early_stopping",
            EarlyStopping(
                monitor="valid_loss",
                patience=CONFIG["early_stopping_patience"],
                lower_is_better=True,
                load_best=True,
            ),
        ),
    ]

    train_generator = None
    if CONFIG["set_seed"]:
        assert fold_seed is not None
        train_generator = torch.Generator()
        train_generator.manual_seed(fold_seed)

    clf_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "max_epochs": CONFIG["n_epochs"],
        "device": DEVICE,
        "callbacks": callbacks,
        "train_split": predefined_split(valid_set),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
    }

    if CONFIG["optimizer"] == "Adam":
        clf_kwargs["optimizer"] = torch.optim.Adam
    elif CONFIG["optimizer"] == "AdamW":
        clf_kwargs["optimizer"] = torch.optim.AdamW
    else:
        raise ValueError(f"Unsupported optimizer: {CONFIG['optimizer']}")
    
    if CONFIG["criterion"] == "CrossEntropyLoss":
        clf_kwargs["criterion"] = torch.nn.CrossEntropyLoss()

    if CONFIG["learning_rate"] is not None:
        clf_kwargs["lr"] = CONFIG["learning_rate"]

    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator

    clf = EEGClassifier(
        model,
        **clf_kwargs,
    )

    clf.fit(train_set, y=None)
    y_pred = clf.predict(test_set)

    stopped_epoch = int(clf.history[-1]["epoch"]) if len(clf.history) > 0 else 0 # type: ignore
    valid_loss_curve = [
        (int(row["epoch"]), float(row["valid_loss"]))
        for row in clf.history
        if "valid_loss" in row
    ]

    if valid_loss_curve:
        best_epoch, best_valid_loss = min(valid_loss_curve, key=lambda x: x[1])  # lower is better
    else:
        best_epoch, best_valid_loss = None, None

    acc = float(accuracy_score(y_test, y_pred))
    bal_acc = float(balanced_accuracy_score(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist()
    pred_hist = np.bincount(y_pred, minlength=TARGET_N_CLASSES).tolist()

    n_folds = CONFIG["cv_folds"]
    val_loss_str = f"{best_valid_loss:.4f}" if best_valid_loss is not None else "N/A"
    print(f"  Fold {fold_id}/{n_folds} | best_epoch={best_epoch} | stop={stopped_epoch} | val_loss={val_loss_str} | acc={acc:.4f} | bal_acc={bal_acc:.4f}")

    return {
        "subject_id": str(subject_id),
        "fold_id": fold_id,
        "n_train": len(train_set),
        "n_val": len(valid_set),
        "n_test": len(test_set),
        "train_class_counts": train_counts.tolist(),
        "val_class_counts": val_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "best_valid_loss": best_valid_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
    }


## 5.2. Subject Cross-Validation Runner

In [37]:
def make_fold_splits(y, n_folds, val_split, n_classes):
    """Create stratified k-fold index splits for one subject."""
    indices = np.arange(len(y))
    skf = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=BASE_SEED if CONFIG["set_seed"] else None,
    )
    folds = []

    for fold_id, (tv_idx, test_idx) in enumerate(skf.split(indices, y), start=1):
        y_tv = y[tv_idx]
        y_test_f = y[test_idx]

        split_seed = BASE_SEED + int(fold_id) if CONFIG["set_seed"] else None # type: ignore
        tr_local_idx, val_local_idx = train_test_split(
            np.arange(len(tv_idx)),
            test_size=val_split,
            stratify=y_tv,
            random_state=split_seed,
        )

        train_idx = tv_idx[tr_local_idx]
        val_idx = tv_idx[val_local_idx]

        if PRINT_FOLD_CLASS_COUNTS:
            train_counts = np.bincount(y[train_idx], minlength=n_classes)
            val_counts = np.bincount(y[val_idx], minlength=n_classes)
            test_counts = np.bincount(y_test_f, minlength=n_classes)
            print(f"  Fold {fold_id} class counts: train={train_counts.tolist()} val={val_counts.tolist()} test={test_counts.tolist()}")

        folds.append(
            {
                "fold_id": fold_id,
                "idx_train": train_idx,
                "idx_val": val_idx,
                "idx_test": test_idx,
            }
        )

    return folds


In [38]:
def run_subject_cv(subject_id, subject_dataset, n_classes, cv_folds, val_split):
    """Run within-subject stratified CV for one subject."""
    y = get_targets_from_dataset(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\nSubject {subject_id}: {len(subject_dataset)} windows | class_counts={counts.tolist()}")

    folds = make_fold_splits(
        y,
        n_folds=cv_folds,
        val_split=val_split,
        n_classes=n_classes,
    )
    results = []

    for fold in folds:
        result = run_single_fold(
            fold["fold_id"],
            subject_id,
            subject_dataset,
            fold["idx_train"],
            fold["idx_val"],
            fold["idx_test"],
        )
        results.append(result)

    mean_acc = float(np.mean([r["accuracy"] for r in results]))
    mean_bal_acc = float(np.mean([r["balanced_accuracy"] for r in results]))
    std_acc = float(np.std([r["accuracy"] for r in results]))
    std_bal_acc = float(np.std([r["balanced_accuracy"] for r in results]))
    print(f"  Subject {subject_id} summary: acc={mean_acc:.4f}±{std_acc:.4f}  bal_acc={mean_bal_acc:.4f}±{std_bal_acc:.4f}")

    return results


## 5.3. Run All Subjects

In [39]:
print("=" * 70)
print("STARTING CROSS-VALIDATION")
print("=" * 70)
print(f"Subjects:      {list(SUBJECT_WINDOWS.keys())}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Batch size:    {CONFIG['batch_size']}")
print(f"Max epochs:    {CONFIG['n_epochs']}")
print(f"Device:        {DEVICE}")
print(f"Seed control:  {CONFIG['set_seed']}")
print(f"Base seed:     {BASE_SEED}")
print("=" * 70)

FOLD_RESULTS = []

for sid, subject_ds in SUBJECT_WINDOWS.items():
    fold_results = run_subject_cv(
        sid,
        subject_ds,
        TARGET_N_CLASSES,
        CONFIG["cv_folds"],
        CONFIG["val_split"],
    )
    FOLD_RESULTS.extend(fold_results)

print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")

[2026-04-10 11:47:20] ======================================================================
[2026-04-10 11:47:20] STARTING CROSS-VALIDATION
[2026-04-10 11:47:20] ======================================================================
[2026-04-10 11:47:20] Subjects:      [48, 49, 50, 51, 52, 53, 54]
[2026-04-10 11:47:20] CV folds:      5
[2026-04-10 11:47:20] Batch size:    32
[2026-04-10 11:47:20] Max epochs:    5000
[2026-04-10 11:47:20] Device:        cpu
[2026-04-10 11:47:20] Seed control:  False
[2026-04-10 11:47:20] Base seed:     None
[2026-04-10 11:47:20] ======================================================================

[2026-04-10 11:47:20] Subject 48: 3960 windows | class_counts=[3300, 660]
[2026-04-10 11:47:20]   Fold 1 class counts: train=[2112, 422] val=[528, 106] test=[660, 132]
[2026-04-10 11:47:20]   Fold 2 class counts: train=[2112, 422] val=[528, 106] test=[660, 132]
[2026-04-10 11:47:20]   Fold 3 class counts: train=[2112, 422] val=[528, 106] test=[660, 132]
[20

# 6. Results

## 6.1. Aggregate Metrics

In [40]:
# Per-subject aggregation
SUBJECT_METRICS = {}

for result in FOLD_RESULTS:
    sid = result["subject_id"]
    if sid not in SUBJECT_METRICS:
        SUBJECT_METRICS[sid] = {"accuracies": [], "balanced_accuracies": []}
    SUBJECT_METRICS[sid]["accuracies"].append(result["accuracy"])
    SUBJECT_METRICS[sid]["balanced_accuracies"].append(result["balanced_accuracy"])

for sid, m in SUBJECT_METRICS.items():
    m["mean_accuracy"] = float(np.mean(m["accuracies"]))
    m["std_accuracy"] = float(np.std(m["accuracies"]))
    m["mean_balanced_accuracy"] = float(np.mean(m["balanced_accuracies"]))
    m["std_balanced_accuracy"] = float(np.std(m["balanced_accuracies"]))

# Global aggregation
all_accs = [r["accuracy"] for r in FOLD_RESULTS]
all_bal_accs = [r["balanced_accuracy"] for r in FOLD_RESULTS]

GLOBAL_METRICS = {
    "mean_accuracy": float(np.mean(all_accs)),
    "std_accuracy": float(np.std(all_accs)),
    "mean_balanced_accuracy": float(np.mean(all_bal_accs)),
    "std_balanced_accuracy": float(np.std(all_bal_accs)),
    "n_subjects": len(SUBJECT_METRICS),
    "n_folds_total": len(FOLD_RESULTS),
}

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)
for sid, m in sorted(SUBJECT_METRICS.items()):
    print(f"  Subject {sid}:  acc={m['mean_accuracy']:.4f}±{m['std_accuracy']:.4f}  "
          f"bal_acc={m['mean_balanced_accuracy']:.4f}±{m['std_balanced_accuracy']:.4f}")
print("-" * 70)
print(f"  OVERALL:      acc={GLOBAL_METRICS['mean_accuracy']:.4f}±{GLOBAL_METRICS['std_accuracy']:.4f}  "
      f"bal_acc={GLOBAL_METRICS['mean_balanced_accuracy']:.4f}±{GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)

[2026-04-10 11:57:53] ======================================================================
[2026-04-10 11:57:53] AGGREGATED RESULTS
[2026-04-10 11:57:53] ======================================================================
[2026-04-10 11:57:53]   Subject 48:  acc=0.9134±0.0138  bal_acc=0.8153±0.0297
[2026-04-10 11:57:53]   Subject 49:  acc=0.9126±0.0108  bal_acc=0.8161±0.0216
[2026-04-10 11:57:53]   Subject 50:  acc=0.8889±0.0051  bal_acc=0.7370±0.0037
[2026-04-10 11:57:53]   Subject 51:  acc=0.8497±0.0038  bal_acc=0.6044±0.0146
[2026-04-10 11:57:53]   Subject 52:  acc=0.9119±0.0095  bal_acc=0.7998±0.0225
[2026-04-10 11:57:53]   Subject 53:  acc=0.9101±0.0112  bal_acc=0.8018±0.0230
[2026-04-10 11:57:53]   Subject 54:  acc=0.8866±0.0085  bal_acc=0.7550±0.0167
[2026-04-10 11:57:53] ----------------------------------------------------------------------
[2026-04-10 11:57:53]   OVERALL:      acc=0.8962±0.0237  bal_acc=0.7613±0.0729
[2026-04-10 11:57:53] =================================

## 6.2. Save Outputs

In [41]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, 'w') as f:
    json.dump(FOLD_RESULTS, f, indent=2)
print(f"CV results saved to:      {cv_results_path}")

subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, 'w') as f:
    json.dump(SUBJECT_METRICS, f, indent=2)
print(f"Subject metrics saved to: {subject_metrics_path}")

global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, 'w') as f:
    json.dump(GLOBAL_METRICS, f, indent=2)
print(f"Global metrics saved to:  {global_metrics_path}")

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "subjects": [str(s) for s in SUBJECT_TRIALS.keys()],
    "model": "SignalJEPA_PreLocal",
    "pretrained_url": CONFIG["pretrained_url"],
    "global_metrics": GLOBAL_METRICS,
}

run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, 'w') as f:
    json.dump(run_metadata, f, indent=2)
print(f"Run metadata saved to:    {run_metadata_path}")

print(f"\nAll artifacts in: {ARTIFACT_DIR}")

[2026-04-10 11:57:53] CV results saved to:      /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning/ERP/20260410_1145_e773995a/cv_results.json
[2026-04-10 11:57:53] Subject metrics saved to: /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning/ERP/20260410_1145_e773995a/subject_metrics.json
[2026-04-10 11:57:53] Global metrics saved to:  /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning/ERP/20260410_1145_e773995a/global_metrics.json
[2026-04-10 11:57:53] Run metadata saved to:    /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning/ERP/20260410_1145_e773995a/run_metadata.json

[2026-04-10 11:57:53] All artifacts in: /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning/ERP/20260410_1145_e773995a


## 6.3. Final Summary

In [42]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:    {RUN_ID}")
print(f"Artifacts: {ARTIFACT_DIR}")
print()
print("Configuration:")
print(f"  Subjects:          {list(SUBJECT_TRIALS.keys())}")
print(f"  Bandpass:          {CONFIG['bandpass_low']}-{CONFIG['bandpass_high']} Hz")
print(f"  Resample:          {CONFIG['sfreq']} Hz")
print(f"  CV folds:          {CONFIG['cv_folds']}")
print(f"  Learning rate:     {CONFIG['learning_rate']}")
print(f"  Batch size:        {CONFIG['batch_size']}")
print(f"  Max epochs:        {CONFIG['n_epochs']}")
print()
print("Results:")
print(f"  Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
print(f"  Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")


[2026-04-10 11:57:53] ======================================================================
[2026-04-10 11:57:53] EXPERIMENT SUMMARY
[2026-04-10 11:57:53] ======================================================================
[2026-04-10 11:57:53] Run ID:    20260410_1145_e773995a
[2026-04-10 11:57:53] Artifacts: /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning/ERP/20260410_1145_e773995a

[2026-04-10 11:57:53] Configuration:
[2026-04-10 11:57:53]   Subjects:          [48, 49, 50, 51, 52, 53, 54]
[2026-04-10 11:57:53]   Bandpass:          0.5-40.0 Hz
[2026-04-10 11:57:53]   Resample:          128 Hz
[2026-04-10 11:57:53]   CV folds:          5
[2026-04-10 11:57:53]   Learning rate:     None
[2026-04-10 11:57:53]   Batch size:        32
[2026-04-10 11:57:53]   Max epochs:        5000

[2026-04-10 11:57:53] Results:
[2026-04-10 11:57:53]   Mean Accuracy:          0.8962 ± 0.0237
[2026-04-10 11:57:53]   Mean Balanced Accuracy: 0.7613 ± 0.0729
